In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
adata = sc.read_h5ad("/Users/apple/Downloads/mono_macro.h5ad")
adata

In [ ]:
adata.shape

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.columns

In [ ]:
adata.var.head()

In [ ]:
adata.obs["Tissue"].value_counts()

In [ ]:
adata.obs["Patient_ID"].value_counts()

In [ ]:
adata.obs["Stage"].value_counts()

In [ ]:
adata.obs["StageGroup"].value_counts()

In [ ]:
adata.obs["cell_type_with_cluster"].value_counts()

In [ ]:
adata.obs.groupby(["Tissue", "cell_type_with_cluster"]).size()

In [ ]:
pd.crosstab(adata.obs["Tissue"], adata.obs["cell_type_with_cluster"])


In [ ]:
# 1. Confirm available metadata columns
adata.obs.columns

In [ ]:
adata.obs.columns

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np


In [ ]:
adata = sc.read_h5ad("/Users/apple/Downloads/mono_macro.h5ad")

In [ ]:
adata

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

outdir = "/Users/apple/Downloads/qc_outputs"
os.makedirs(outdir, exist_ok=True)

qc_cols = ["nCount_RNA", "nFeature_RNA", "percent.mt", "percent.hb"]
key_cols = qc_cols + ["Tissue", "Patient_ID", "StageGroup", "cell_type_with_cluster"]

# 1. metadata columns
pd.Series(adata.obs.columns).to_csv(f"{outdir}/obs_columns.csv", index=False)

# 2. overall QC summary
adata.obs[qc_cols].describe().T.to_csv(f"{outdir}/qc_overall_summary.csv")

# 3. missing value check
adata.obs[key_cols].isna().sum().to_csv(f"{outdir}/missing_values.csv")

# 4. QC by tissue
adata.obs.groupby("Tissue")[qc_cols].describe().to_csv(f"{outdir}/qc_by_tissue.csv")

# 5. QC by cell type
adata.obs.groupby("cell_type_with_cluster")[qc_cols].describe().to_csv(f"{outdir}/qc_by_cell_type.csv")

# 6. doublet-related columns
doublet_cols = [col for col in adata.obs.columns if "doublet" in col.lower()]
pd.Series(doublet_cols).to_csv(f"{outdir}/doublet_columns.csv", index=False)

# 7. raw/layers/obsm/uns summary
with open(f"{outdir}/object_structure.txt", "w") as f:
    f.write("adata shape:\n")
    f.write(str(adata.shape) + "\n\n")
    
    f.write("adata.raw:\n")
    f.write(str(adata.raw) + "\n\n")
    
    f.write("adata.layers.keys():\n")
    f.write(str(list(adata.layers.keys())) + "\n\n")
    
    f.write("adata.obsm.keys():\n")
    f.write(str(list(adata.obsm.keys())) + "\n\n")
    
    f.write("adata.uns.keys():\n")
    f.write(str(list(adata.uns.keys())) + "\n\n")
    
    f.write("adata.X type:\n")
    f.write(str(type(adata.X)) + "\n\n")
    
    if hasattr(adata.X, "data"):
        f.write("adata.X first 20 non-zero values:\n")
        f.write(str(adata.X.data[:20]) + "\n")
        f.write("\nadata.X min/max:\n")
        f.write(str((adata.X.data.min(), adata.X.data.max())) + "\n")
    else:
        f.write("adata.X first 5x5 values:\n")
        f.write(str(adata.X[:5, :5]) + "\n")
        f.write("\nadta.X min/max:\n")
        f.write(str((adata.X.min(), adata.X.max())) + "\n")

In [ ]:
sc.pl.violin(
    adata,
    ["nCount_RNA", "nFeature_RNA", "percent.mt", "percent.hb"],
    groupby="Tissue",
    rotation=45,
    show=False
)
plt.savefig(f"{outdir}/qc_violin_by_tissue.png", dpi=300, bbox_inches="tight")
plt.close()

sc.pl.violin(
    adata,
    ["nCount_RNA", "nFeature_RNA", "percent.mt", "percent.hb"],
    groupby="cell_type_with_cluster",
    rotation=45,
    show=False
)
plt.savefig(f"{outdir}/qc_violin_by_cell_type.png", dpi=300, bbox_inches="tight")
plt.close()

sc.pl.scatter(adata, x="nCount_RNA", y="nFeature_RNA", show=False)
plt.savefig(f"{outdir}/scatter_counts_features.png", dpi=300, bbox_inches="tight")
plt.close()

sc.pl.scatter(adata, x="nCount_RNA", y="percent.mt", show=False)
plt.savefig(f"{outdir}/scatter_counts_mt.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

# 新建 2.5 专用输出文件夹，避免和 qc_outputs 混淆
outdir = "/Users/apple/Downloads/section2_5_outputs"
os.makedirs(outdir, exist_ok=True)

# 去掉没有实际细胞的 unused categories，避免图和表里出现 0-count cell types
if "cell_type_with_cluster" in adata.obs.columns:
    if hasattr(adata.obs["cell_type_with_cluster"], "cat"):
        adata.obs["cell_type_with_cluster"] = adata.obs["cell_type_with_cluster"].cat.remove_unused_categories()

# 1. 保存 object structure，用于判断已有 PCA / Harmony / UMAP
with open(f"{outdir}/object_structure_2_5.txt", "w") as f:
    f.write("adata shape:\n")
    f.write(str(adata.shape) + "\n\n")
    
    f.write("adata.obsm.keys():\n")
    f.write(str(list(adata.obsm.keys())) + "\n\n")
    
    f.write("adata.uns.keys():\n")
    f.write(str(list(adata.uns.keys())) + "\n\n")
    
    f.write("adata.obs.columns:\n")
    f.write(str(list(adata.obs.columns)) + "\n\n")

# 2. 保存 cell-type annotation 数量
adata.obs["cell_type_with_cluster"].value_counts().to_csv(
    f"{outdir}/cell_type_counts.csv"
)

# 3. 保存 Tissue / StageGroup 与 cell type 的交叉表
pd.crosstab(
    adata.obs["Tissue"],
    adata.obs["cell_type_with_cluster"]
).to_csv(f"{outdir}/cell_type_by_tissue.csv")

pd.crosstab(
    adata.obs["StageGroup"],
    adata.obs["cell_type_with_cluster"]
).to_csv(f"{outdir}/cell_type_by_stagegroup.csv")

# 4. UMAP by cell type
sc.pl.umap(
    adata,
    color="cell_type_with_cluster",
    show=False
)
plt.savefig(f"{outdir}/umap_by_cell_type.png", dpi=300, bbox_inches="tight")
plt.close()

# 5. UMAP by tissue
sc.pl.umap(
    adata,
    color="Tissue",
    show=False
)
plt.savefig(f"{outdir}/umap_by_tissue.png", dpi=300, bbox_inches="tight")
plt.close()

# 6. UMAP by fibrosis-stage group
sc.pl.umap(
    adata,
    color="StageGroup",
    show=False
)
plt.savefig(f"{outdir}/umap_by_stagegroup.png", dpi=300, bbox_inches="tight")
plt.close()

# 7. 检查 marker genes 是否存在
marker_genes = [
    # general myeloid / monocyte markers
    "LYZ", "LST1", "TYROBP", "AIF1",
    
    # CD14 monocyte markers
    "CD14", "FCN1", "S100A8", "S100A9", "VCAN", "CCR2",
    
    # CD16 monocyte markers
    "FCGR3A", "MS4A7", "LILRB1", "CX3CR1",
    
    # macrophage / Kupffer-like / tissue macrophage markers
    "C1QA", "C1QB", "C1QC", "APOE", "APOC1", "MRC1", "MARCO", "TIMD4",
    
    # disease-associated / scar-associated macrophage markers
    "TREM2", "CD9", "SPP1", "GPNMB", "LGALS3"
]

# 去重但保留顺序
marker_genes = list(dict.fromkeys(marker_genes))

present_markers = [g for g in marker_genes if g in adata.var_names]
missing_markers = [g for g in marker_genes if g not in adata.var_names]

pd.DataFrame({
    "present_markers": pd.Series(present_markers)
}).to_csv(f"{outdir}/present_marker_genes.csv", index=False)

pd.DataFrame({
    "missing_markers": pd.Series(missing_markers)
}).to_csv(f"{outdir}/missing_marker_genes.csv", index=False)

with open(f"{outdir}/marker_gene_summary.txt", "w") as f:
    f.write("Present markers:\n")
    f.write(str(present_markers) + "\n\n")
    f.write("Missing markers:\n")
    f.write(str(missing_markers) + "\n")

# 8. 保存 marker gene UMAP 图，分批保存，避免一张图太挤
batch_size = 6

for i in range(0, len(present_markers), batch_size):
    batch = present_markers[i:i + batch_size]
    sc.pl.umap(
        adata,
        color=batch,
        use_raw=False,
        show=False
    )
    plt.savefig(
        f"{outdir}/umap_marker_genes_batch_{i//batch_size + 1}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

# 9. marker gene dotplot by cell type
if len(present_markers) > 0:
    dp = sc.pl.dotplot(
        adata,
        var_names=present_markers,
        groupby="cell_type_with_cluster",
        standard_scale="var",
        use_raw=False,
        return_fig=True
    )
    dp.savefig(f"{outdir}/dotplot_markers_by_cell_type.png")

# 10. marker gene matrixplot by cell type
if len(present_markers) > 0:
    mp = sc.pl.matrixplot(
        adata,
        var_names=present_markers,
        groupby="cell_type_with_cluster",
        standard_scale="var",
        use_raw=False,
        return_fig=True
    )
    mp.savefig(f"{outdir}/matrixplot_markers_by_cell_type.png")

print("All 2.5 outputs saved to:")
print(outdir)
print("Files generated:")
for file in os.listdir(outdir):
    print(file)

In [ ]:
%pip install leidenalg igraph

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad("/Users/apple/Downloads/mono_macro.h5ad")

In [ ]:
adata

In [ ]:
%pip install leidenalg igraph

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

# 新建 2.6 专用输出文件夹，避免和之前混淆
outdir = "/Users/apple/Downloads/section2_6_outputs"
os.makedirs(outdir, exist_ok=True)

# 确认 adata 已经存在
try:
    adata
except NameError:
    raise NameError("当前 Jupyter 里没有 adata。请先读取 mono_macro.h5ad 后再运行本段代码。")

# 去掉没有实际细胞的 unused categories
for col in ["cell_type_with_cluster", "Tissue", "StageGroup"]:
    if col in adata.obs.columns and hasattr(adata.obs[col], "cat"):
        adata.obs[col] = adata.obs[col].cat.remove_unused_categories()

# 选择用于 subclustering 的低维表示
if "X_harmony" in adata.obsm.keys():
    use_rep = "X_harmony"
elif "X_pca" in adata.obsm.keys():
    use_rep = "X_pca"
else:
    raise ValueError("未找到 X_harmony 或 X_pca，不能直接进行 neighbour graph / Leiden subclustering。")

# 1. 构建 neighbour graph
sc.pp.neighbors(
    adata,
    use_rep=use_rep,
    n_neighbors=15,
    key_added="neighbors_subcluster"
)

# 2. 多个 Leiden resolution，用于比较 subcluster 数量
resolutions = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

for res in resolutions:
    key = f"leiden_{res}"
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        neighbors_key="neighbors_subcluster"
    )

# 3. 保存不同 resolution 的 cluster 数量
cluster_summary = {}

for res in resolutions:
    key = f"leiden_{res}"
    cluster_summary[key] = adata.obs[key].value_counts().sort_index()

cluster_summary_df = pd.DataFrame(cluster_summary).fillna(0).astype(int)
cluster_summary_df.to_csv(f"{outdir}/leiden_resolution_cluster_counts.csv")

# 4. 暂定 leiden_0.5 作为主要 subcluster 分析结果
# 后面如果发现 cluster 太多/太少，可以改成 leiden_0.4 或 leiden_0.6
final_cluster_key = "leiden_0.5"

# 5. 保存 final subcluster 的基础数量
adata.obs[final_cluster_key].value_counts().sort_index().to_csv(
    f"{outdir}/final_subcluster_counts.csv"
)

# 6. subcluster × broad cell type
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["cell_type_with_cluster"]
).to_csv(f"{outdir}/subcluster_by_cell_type_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["cell_type_with_cluster"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_cell_type_percent.csv")

# 7. subcluster × tissue
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["Tissue"]
).to_csv(f"{outdir}/subcluster_by_tissue_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["Tissue"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_tissue_percent.csv")

# 8. subcluster × StageGroup
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["StageGroup"]
).to_csv(f"{outdir}/subcluster_by_stagegroup_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["StageGroup"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_stagegroup_percent.csv")

# 9. UMAP by different Leiden resolutions
for res in resolutions:
    key = f"leiden_{res}"
    sc.pl.umap(
        adata,
        color=key,
        legend_loc="right margin",
        show=False
    )
    plt.savefig(f"{outdir}/umap_{key}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 10. UMAP by final subcluster + existing annotations
sc.pl.umap(
    adata,
    color=[final_cluster_key, "cell_type_with_cluster", "Tissue", "StageGroup"],
    show=False
)
plt.savefig(f"{outdir}/umap_final_subcluster_context.png", dpi=300, bbox_inches="tight")
plt.close()

# 11. marker genes for subcluster interpretation
marker_genes = [
    # general myeloid / monocyte markers
    "LYZ", "LST1", "TYROBP", "AIF1",
    
    # CD14 monocyte markers
    "CD14", "FCN1", "S100A8", "S100A9", "VCAN", "CCR2",
    
    # CD16 monocyte markers
    "FCGR3A", "MS4A7", "LILRB1", "CX3CR1",
    
    # macrophage / tissue macrophage markers
    "C1QA", "C1QB", "C1QC", "APOE", "APOC1", "MRC1", "MARCO", "TIMD4",
    
    # disease-associated / scar-associated macrophage markers
    "TREM2", "CD9", "SPP1", "GPNMB", "LGALS3"
]

marker_genes = list(dict.fromkeys(marker_genes))
present_markers = [g for g in marker_genes if g in adata.var_names]
missing_markers = [g for g in marker_genes if g not in adata.var_names]

pd.Series(present_markers).to_csv(f"{outdir}/present_markers_2_6.csv", index=False)
pd.Series(missing_markers).to_csv(f"{outdir}/missing_markers_2_6.csv", index=False)

# 12. dotplot and matrixplot by final subcluster
dp = sc.pl.dotplot(
    adata,
    var_names=present_markers,
    groupby=final_cluster_key,
    standard_scale="var",
    use_raw=False,
    return_fig=True
)
dp.savefig(f"{outdir}/dotplot_markers_by_final_subcluster.png")

mp = sc.pl.matrixplot(
    adata,
    var_names=present_markers,
    groupby=final_cluster_key,
    standard_scale="var",
    use_raw=False,
    return_fig=True
)
mp.savefig(f"{outdir}/matrixplot_markers_by_final_subcluster.png")

# 13. rank marker genes per final subcluster
# 这一步可能需要几分钟
sc.tl.rank_genes_groups(
    adata,
    groupby=final_cluster_key,
    method="wilcoxon",
    use_raw=False,
    n_genes=50
)

ranked_markers_df = sc.get.rank_genes_groups_df(adata, group=None)
ranked_markers_df.to_csv(f"{outdir}/ranked_marker_genes_by_final_subcluster.csv", index=False)

# 14. 保存 top 10 markers per subcluster，方便快速查看
top10_markers = (
    ranked_markers_df
    .groupby("group")
    .head(10)
    .reset_index(drop=True)
)
top10_markers.to_csv(f"{outdir}/top10_marker_genes_by_final_subcluster.csv", index=False)

# 15. rank genes dotplot
rg = sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    groupby=final_cluster_key,
    standard_scale="var",
    show=False,
    return_fig=True
)
rg.savefig(f"{outdir}/rank_genes_groups_dotplot_top5.png")

# 16. gene signature scores
signatures = {
    "CD14_monocyte_score": ["CD14", "FCN1", "S100A8", "S100A9", "VCAN", "CCR2"],
    "CD16_monocyte_score": ["FCGR3A", "MS4A7", "LILRB1", "CX3CR1"],
    "Macrophage_complement_score": ["C1QA", "C1QB", "C1QC", "APOE", "APOC1"],
    "Disease_associated_macrophage_score": ["TREM2", "CD9", "SPP1", "GPNMB", "LGALS3"]
}

score_names = []

for score_name, genes in signatures.items():
    genes_present = [g for g in genes if g in adata.var_names]
    if len(genes_present) >= 2:
        sc.tl.score_genes(
            adata,
            gene_list=genes_present,
            score_name=score_name,
            use_raw=False
        )
        score_names.append(score_name)

# 17. 保存 signature score summaries
if len(score_names) > 0:
    adata.obs.groupby(final_cluster_key)[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_final_subcluster.csv"
    )
    
    adata.obs.groupby("cell_type_with_cluster")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_cell_type.csv"
    )
    
    adata.obs.groupby("Tissue")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_tissue.csv"
    )
    
    adata.obs.groupby("StageGroup")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_stagegroup.csv"
    )
    
    sc.pl.umap(
        adata,
        color=score_names,
        show=False
    )
    plt.savefig(f"{outdir}/umap_signature_scores.png", dpi=300, bbox_inches="tight")
    plt.close()

# 18. 保存 object summary
with open(f"{outdir}/section2_6_summary.txt", "w") as f:
    f.write("Section 2.6 monocyte/macrophage analysis summary\n\n")
    
    f.write("Used representation for neighbours:\n")
    f.write(str(use_rep) + "\n\n")
    
    f.write("Neighbour graph:\n")
    f.write("n_neighbors = 15\n")
    f.write("neighbors_key = neighbors_subcluster\n\n")
    
    f.write("Leiden resolutions tested:\n")
    f.write(str(resolutions) + "\n\n")
    
    f.write("Final cluster key used for downstream summaries:\n")
    f.write(final_cluster_key + "\n\n")
    
    f.write("Present marker genes:\n")
    f.write(str(present_markers) + "\n\n")
    
    f.write("Missing marker genes:\n")
    f.write(str(missing_markers) + "\n\n")
    
    f.write("Signature scores calculated:\n")
    f.write(str(score_names) + "\n\n")
    
    f.write("adata.obsm.keys():\n")
    f.write(str(list(adata.obsm.keys())) + "\n\n")
    
    f.write("adata.uns.keys():\n")
    f.write(str(list(adata.uns.keys())) + "\n\n")
    
    f.write("adata.obs columns added:\n")
    added_cols = [col for col in adata.obs.columns if col.startswith("leiden_") or col in score_names]
    f.write(str(added_cols) + "\n")

# 19. 保存带 subcluster 和 signature scores 的 h5ad，方便之后继续分析
adata.write_h5ad(f"{outdir}/mono_macro_with_subclusters_2_6.h5ad")

print("All 2.6 outputs saved to:")
print(outdir)
print("\nFiles generated:")
for file in os.listdir(outdir):
    print(file)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

# 新建 2.6 专用输出文件夹，避免和之前混淆
outdir = "/Users/apple/Downloads/section2_6_outputs"
os.makedirs(outdir, exist_ok=True)

# 确认 adata 已经存在
try:
    adata
except NameError:
    raise NameError("当前 Jupyter 里没有 adata。请先读取 mono_macro.h5ad 后再运行本段代码。")

# 去掉没有实际细胞的 unused categories
for col in ["cell_type_with_cluster", "Tissue", "StageGroup"]:
    if col in adata.obs.columns and hasattr(adata.obs[col], "cat"):
        adata.obs[col] = adata.obs[col].cat.remove_unused_categories()

# 选择用于 subclustering 的低维表示
if "X_harmony" in adata.obsm.keys():
    use_rep = "X_harmony"
elif "X_pca" in adata.obsm.keys():
    use_rep = "X_pca"
else:
    raise ValueError("未找到 X_harmony 或 X_pca，不能直接进行 neighbour graph / Leiden subclustering。")

# 1. 构建 neighbour graph
sc.pp.neighbors(
    adata,
    use_rep=use_rep,
    n_neighbors=15,
    key_added="neighbors_subcluster"
)

# 2. 多个 Leiden resolution，用于比较 subcluster 数量
resolutions = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

for res in resolutions:
    key = f"leiden_{res}"
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        neighbors_key="neighbors_subcluster"
    )

# 3. 保存不同 resolution 的 cluster 数量
cluster_summary = {}

for res in resolutions:
    key = f"leiden_{res}"
    cluster_summary[key] = adata.obs[key].value_counts().sort_index()

cluster_summary_df = pd.DataFrame(cluster_summary).fillna(0).astype(int)
cluster_summary_df.to_csv(f"{outdir}/leiden_resolution_cluster_counts.csv")

# 4. 暂定 leiden_0.5 作为主要 subcluster 分析结果
# 后面如果发现 cluster 太多/太少，可以改成 leiden_0.4 或 leiden_0.6
final_cluster_key = "leiden_0.5"

# 5. 保存 final subcluster 的基础数量
adata.obs[final_cluster_key].value_counts().sort_index().to_csv(
    f"{outdir}/final_subcluster_counts.csv"
)

# 6. subcluster × broad cell type
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["cell_type_with_cluster"]
).to_csv(f"{outdir}/subcluster_by_cell_type_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["cell_type_with_cluster"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_cell_type_percent.csv")

# 7. subcluster × tissue
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["Tissue"]
).to_csv(f"{outdir}/subcluster_by_tissue_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["Tissue"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_tissue_percent.csv")

# 8. subcluster × StageGroup
pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["StageGroup"]
).to_csv(f"{outdir}/subcluster_by_stagegroup_counts.csv")

(pd.crosstab(
    adata.obs[final_cluster_key],
    adata.obs["StageGroup"],
    normalize="index"
) * 100).to_csv(f"{outdir}/subcluster_by_stagegroup_percent.csv")

# 9. UMAP by different Leiden resolutions
for res in resolutions:
    key = f"leiden_{res}"
    sc.pl.umap(
        adata,
        color=key,
        legend_loc="right margin",
        show=False
    )
    plt.savefig(f"{outdir}/umap_{key}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 10. UMAP by final subcluster + existing annotations
sc.pl.umap(
    adata,
    color=[final_cluster_key, "cell_type_with_cluster", "Tissue", "StageGroup"],
    show=False
)
plt.savefig(f"{outdir}/umap_final_subcluster_context.png", dpi=300, bbox_inches="tight")
plt.close()

# 11. marker genes for subcluster interpretation
marker_genes = [
    # general myeloid / monocyte markers
    "LYZ", "LST1", "TYROBP", "AIF1",
    
    # CD14 monocyte markers
    "CD14", "FCN1", "S100A8", "S100A9", "VCAN", "CCR2",
    
    # CD16 monocyte markers
    "FCGR3A", "MS4A7", "LILRB1", "CX3CR1",
    
    # macrophage / tissue macrophage markers
    "C1QA", "C1QB", "C1QC", "APOE", "APOC1", "MRC1", "MARCO", "TIMD4",
    
    # disease-associated / scar-associated macrophage markers
    "TREM2", "CD9", "SPP1", "GPNMB", "LGALS3"
]

marker_genes = list(dict.fromkeys(marker_genes))
present_markers = [g for g in marker_genes if g in adata.var_names]
missing_markers = [g for g in marker_genes if g not in adata.var_names]

pd.Series(present_markers).to_csv(f"{outdir}/present_markers_2_6.csv", index=False)
pd.Series(missing_markers).to_csv(f"{outdir}/missing_markers_2_6.csv", index=False)

# 12. dotplot and matrixplot by final subcluster
dp = sc.pl.dotplot(
    adata,
    var_names=present_markers,
    groupby=final_cluster_key,
    standard_scale="var",
    use_raw=False,
    return_fig=True
)
dp.savefig(f"{outdir}/dotplot_markers_by_final_subcluster.png")

mp = sc.pl.matrixplot(
    adata,
    var_names=present_markers,
    groupby=final_cluster_key,
    standard_scale="var",
    use_raw=False,
    return_fig=True
)
mp.savefig(f"{outdir}/matrixplot_markers_by_final_subcluster.png")

# 13. rank marker genes per final subcluster
# 这一步可能需要几分钟
sc.tl.rank_genes_groups(
    adata,
    groupby=final_cluster_key,
    method="wilcoxon",
    use_raw=False,
    n_genes=50
)

ranked_markers_df = sc.get.rank_genes_groups_df(adata, group=None)
ranked_markers_df.to_csv(f"{outdir}/ranked_marker_genes_by_final_subcluster.csv", index=False)

# 14. 保存 top 10 markers per subcluster，方便快速查看
top10_markers = (
    ranked_markers_df
    .groupby("group")
    .head(10)
    .reset_index(drop=True)
)
top10_markers.to_csv(f"{outdir}/top10_marker_genes_by_final_subcluster.csv", index=False)

# 15. rank genes dotplot
rg = sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    groupby=final_cluster_key,
    standard_scale="var",
    show=False,
    return_fig=True
)
rg.savefig(f"{outdir}/rank_genes_groups_dotplot_top5.png")

# 16. gene signature scores
signatures = {
    "CD14_monocyte_score": ["CD14", "FCN1", "S100A8", "S100A9", "VCAN", "CCR2"],
    "CD16_monocyte_score": ["FCGR3A", "MS4A7", "LILRB1", "CX3CR1"],
    "Macrophage_complement_score": ["C1QA", "C1QB", "C1QC", "APOE", "APOC1"],
    "Disease_associated_macrophage_score": ["TREM2", "CD9", "SPP1", "GPNMB", "LGALS3"]
}

score_names = []

for score_name, genes in signatures.items():
    genes_present = [g for g in genes if g in adata.var_names]
    if len(genes_present) >= 2:
        sc.tl.score_genes(
            adata,
            gene_list=genes_present,
            score_name=score_name,
            use_raw=False
        )
        score_names.append(score_name)

# 17. 保存 signature score summaries
if len(score_names) > 0:
    adata.obs.groupby(final_cluster_key)[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_final_subcluster.csv"
    )
    
    adata.obs.groupby("cell_type_with_cluster")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_cell_type.csv"
    )
    
    adata.obs.groupby("Tissue")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_tissue.csv"
    )
    
    adata.obs.groupby("StageGroup")[score_names].mean().to_csv(
        f"{outdir}/signature_scores_by_stagegroup.csv"
    )
    
    sc.pl.umap(
        adata,
        color=score_names,
        show=False
    )
    plt.savefig(f"{outdir}/umap_signature_scores.png", dpi=300, bbox_inches="tight")
    plt.close()

# 18. 保存 object summary
with open(f"{outdir}/section2_6_summary.txt", "w") as f:
    f.write("Section 2.6 monocyte/macrophage analysis summary\n\n")
    
    f.write("Used representation for neighbours:\n")
    f.write(str(use_rep) + "\n\n")
    
    f.write("Neighbour graph:\n")
    f.write("n_neighbors = 15\n")
    f.write("neighbors_key = neighbors_subcluster\n\n")
    
    f.write("Leiden resolutions tested:\n")
    f.write(str(resolutions) + "\n\n")
    
    f.write("Final cluster key used for downstream summaries:\n")
    f.write(final_cluster_key + "\n\n")
    
    f.write("Present marker genes:\n")
    f.write(str(present_markers) + "\n\n")
    
    f.write("Missing marker genes:\n")
    f.write(str(missing_markers) + "\n\n")
    
    f.write("Signature scores calculated:\n")
    f.write(str(score_names) + "\n\n")
    
    f.write("adata.obsm.keys():\n")
    f.write(str(list(adata.obsm.keys())) + "\n\n")
    
    f.write("adata.uns.keys():\n")
    f.write(str(list(adata.uns.keys())) + "\n\n")
    
    f.write("adata.obs columns added:\n")
    added_cols = [col for col in adata.obs.columns if col.startswith("leiden_") or col in score_names]
    f.write(str(added_cols) + "\n")

# 19. 保存带 subcluster 和 signature scores 的 h5ad，方便之后继续分析
adata.write_h5ad(f"{outdir}/mono_macro_with_subclusters_2_6.h5ad")

print("All 2.6 outputs saved to:")
print(outdir)
print("\nFiles generated:")
for file in os.listdir(outdir):
    print(file)

In [ ]:
import os

outdir = "/Users/apple/Downloads/section2_6_outputs"

for file in os.listdir(outdir):
    print(file)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

outdir = "/Users/apple/Downloads/section2_7_outputs"
os.makedirs(outdir, exist_ok=True)

# 确认 final cluster key
final_cluster_key = "leiden_0.5"

# 确认必要列存在
required_cols = ["Patient_ID", "Tissue", "StageGroup", "cell_type_with_cluster", final_cluster_key]
missing_cols = [col for col in required_cols if col not in adata.obs.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing required columns: {missing_cols}")

# 1. Basic statistical analysis summary
with open(f"{outdir}/section2_7_summary.txt", "w") as f:
    f.write("Section 2.7 statistical analysis summary\n\n")
    f.write("Final subcluster key:\n")
    f.write(final_cluster_key + "\n\n")
    f.write("Number of cells:\n")
    f.write(str(adata.n_obs) + "\n\n")
    f.write("Number of genes:\n")
    f.write(str(adata.n_vars) + "\n\n")
    f.write("Number of Patient_ID values:\n")
    f.write(str(adata.obs["Patient_ID"].nunique()) + "\n\n")
    f.write("Tissue categories:\n")
    f.write(str(list(adata.obs["Tissue"].unique())) + "\n\n")
    f.write("StageGroup categories:\n")
    f.write(str(list(adata.obs["StageGroup"].unique())) + "\n\n")
    f.write("Cell-type categories:\n")
    f.write(str(list(adata.obs["cell_type_with_cluster"].unique())) + "\n\n")

# 2. Tissue × StageGroup balance
pd.crosstab(
    adata.obs["Tissue"],
    adata.obs["StageGroup"]
).to_csv(f"{outdir}/tissue_by_stagegroup_counts.csv")

(pd.crosstab(
    adata.obs["Tissue"],
    adata.obs["StageGroup"],
    normalize="index"
) * 100).to_csv(f"{outdir}/tissue_by_stagegroup_percent_by_tissue.csv")

# 3. Patient/sample × Tissue
pd.crosstab(
    adata.obs["Patient_ID"],
    adata.obs["Tissue"]
).to_csv(f"{outdir}/patient_by_tissue_counts.csv")

# 4. Patient/sample × StageGroup
pd.crosstab(
    adata.obs["Patient_ID"],
    adata.obs["StageGroup"]
).to_csv(f"{outdir}/patient_by_stagegroup_counts.csv")

# 5. Patient-level cell-type composition
patient_celltype_counts = (
    adata.obs
    .groupby(["Patient_ID", "Tissue", "StageGroup", "cell_type_with_cluster"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

patient_celltype_counts.to_csv(
    f"{outdir}/patient_tissue_stage_celltype_counts.csv",
    index=False
)

patient_celltype_total = (
    patient_celltype_counts
    .groupby(["Patient_ID", "Tissue", "StageGroup"], observed=True)["n_cells"]
    .transform("sum")
)

patient_celltype_counts["percent_within_patient_tissue"] = (
    patient_celltype_counts["n_cells"] / patient_celltype_total * 100
)

patient_celltype_counts.to_csv(
    f"{outdir}/patient_tissue_stage_celltype_percent.csv",
    index=False
)

# 6. Patient-level subcluster composition
patient_subcluster_counts = (
    adata.obs
    .groupby(["Patient_ID", "Tissue", "StageGroup", final_cluster_key], observed=True)
    .size()
    .reset_index(name="n_cells")
)

patient_subcluster_counts.to_csv(
    f"{outdir}/patient_tissue_stage_subcluster_counts.csv",
    index=False
)

patient_subcluster_total = (
    patient_subcluster_counts
    .groupby(["Patient_ID", "Tissue", "StageGroup"], observed=True)["n_cells"]
    .transform("sum")
)

patient_subcluster_counts["percent_within_patient_tissue"] = (
    patient_subcluster_counts["n_cells"] / patient_subcluster_total * 100
)

patient_subcluster_counts.to_csv(
    f"{outdir}/patient_tissue_stage_subcluster_percent.csv",
    index=False
)

# 7. Signature score summaries if scores exist
score_names = [
    "CD14_monocyte_score",
    "CD16_monocyte_score",
    "Macrophage_complement_score",
    "Disease_associated_macrophage_score"
]

score_names = [s for s in score_names if s in adata.obs.columns]

if len(score_names) > 0:
    # Cell-level summaries
    adata.obs.groupby("Tissue", observed=True)[score_names].describe().to_csv(
        f"{outdir}/signature_scores_by_tissue_describe.csv"
    )
    
    adata.obs.groupby("StageGroup", observed=True)[score_names].describe().to_csv(
        f"{outdir}/signature_scores_by_stagegroup_describe.csv"
    )
    
    adata.obs.groupby(final_cluster_key, observed=True)[score_names].describe().to_csv(
        f"{outdir}/signature_scores_by_subcluster_describe.csv"
    )
    
    # Patient/sample-level mean signature scores
    patient_signature_scores = (
        adata.obs
        .groupby(["Patient_ID", "Tissue", "StageGroup"], observed=True)[score_names]
        .mean()
        .reset_index()
    )
    
    patient_signature_scores.to_csv(
        f"{outdir}/patient_tissue_stage_signature_scores_mean.csv",
        index=False
    )

# 8. Simple non-parametric comparisons at patient/sample level
# Tissue comparisons: Kruskal-Wallis across tissue groups
test_results = []

if len(score_names) > 0:
    for score in score_names:
        temp = patient_signature_scores.dropna(subset=[score])
        
        groups = [
            group[score].values
            for _, group in temp.groupby("Tissue", observed=True)
            if len(group[score].dropna()) > 1
        ]
        
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            test_results.append({
                "comparison": "Tissue",
                "variable": score,
                "test": "Kruskal-Wallis",
                "statistic": stat,
                "p_value": p
            })
        
        # StageGroup comparisons
        groups = [
            group[score].values
            for _, group in temp.groupby("StageGroup", observed=True)
            if len(group[score].dropna()) > 1
        ]
        
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            test_results.append({
                "comparison": "StageGroup",
                "variable": score,
                "test": "Kruskal-Wallis",
                "statistic": stat,
                "p_value": p
            })

test_results_df = pd.DataFrame(test_results)

if len(test_results_df) > 0:
    # Benjamini-Hochberg FDR correction
    test_results_df = test_results_df.sort_values("p_value").reset_index(drop=True)
    m = len(test_results_df)
    test_results_df["rank"] = np.arange(1, m + 1)
    test_results_df["fdr_bh"] = test_results_df["p_value"] * m / test_results_df["rank"]
    test_results_df["fdr_bh"] = test_results_df["fdr_bh"].clip(upper=1)
    test_results_df.to_csv(f"{outdir}/signature_score_statistical_tests.csv", index=False)

# 9. Save top ranked marker genes from existing rank_genes_groups if available
if "rank_genes_groups" in adata.uns.keys():
    ranked_df = sc.get.rank_genes_groups_df(adata, group=None)
    ranked_df.to_csv(f"{outdir}/ranked_marker_genes_from_existing_rank_genes_groups.csv", index=False)

# 10. Figures: patient/sample-level signature scores
if len(score_names) > 0:
    for score in score_names:
        plt.figure(figsize=(6, 4))
        patient_signature_scores.boxplot(column=score, by="Tissue", rot=45)
        plt.title(score + " by Tissue")
        plt.suptitle("")
        plt.ylabel(score)
        plt.tight_layout()
        plt.savefig(f"{outdir}/boxplot_{score}_by_tissue.png", dpi=300, bbox_inches="tight")
        plt.close()
        
        plt.figure(figsize=(6, 4))
        patient_signature_scores.boxplot(column=score, by="StageGroup", rot=45)
        plt.title(score + " by StageGroup")
        plt.suptitle("")
        plt.ylabel(score)
        plt.tight_layout()
        plt.savefig(f"{outdir}/boxplot_{score}_by_stagegroup.png", dpi=300, bbox_inches="tight")
        plt.close()

print("All 2.7 outputs saved to:")
print(outdir)
print("\nFiles generated:")
for file in os.listdir(outdir):
    print(file)

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata


In [ ]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print("obs columns:", adata.obs.columns.tolist())
    for c in adata.obs.columns:
        if adata.obs[c].nunique(dropna=False) <= 30:
            print(f"\n[{c}]  n_unique={adata.obs[c].nunique(dropna=False)}")
            print(adata.obs[c].value_counts(dropna=False))
    print("\nlayers:", list(adata.layers.keys()))
    print("obsm:", list(adata.obsm.keys()))
    print("uns keys:", list(adata.uns.keys()))

text = buf.getvalue()
print(text)  # 同时在 notebook 里显示

with open("/Users/apple/Downloads/section2_6_outputs/step0_columns.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("\n已保存到: /Users/apple/Downloads/section2_6_outputs/step0_columns.txt")

In [ ]:
import io, contextlib
import numpy as np, pandas as pd, scipy.sparse as sp

PATIENT_COL  = "Patient_ID"
SAMPLE_COL   = "Sample"
TISSUE_COL   = "Tissue"
FIBROSIS_COL = "StageGroup"
STAGESEP_COL = "StageSep"
CELLTYPE_COL = "cell_type_with_cluster"

def densify(M, n=None):
    if n is not None: M = M[:n]
    return M.toarray() if sp.issparse(M) else np.asarray(M)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):

    print("="*60, "\nSTEP 1  identity of .X\n", "="*60)
    Xs = densify(adata.X, 1000)
    print("min:", Xs.min(), "max:", Xs.max())
    print("has negatives (=> scaled/z-scored):", bool((Xs < 0).any()))
    print("all ~integer (=> raw counts):", bool(np.allclose(Xs, np.round(Xs))))
    print("expm1(.X) row sums (first 8):", np.round(np.expm1(Xs).sum(1)[:8], 1))
    print("   -> if ~constant (e.g. ~1e4), .X is log1p-normalised")
    for ln in adata.layers:
        L = densify(adata.layers[ln], 1000)
        print(f"layer '{ln}': min {L.min():.3f} max {L.max():.3f} integer={np.allclose(L, np.round(L))}")

    print("\n"+"="*60, "\nSTEP 2  marker genes present\n", "="*60)
    sigs = {"CD14":["CD14","FCN1","S100A8","S100A9","VCAN","CCR2"],
            "CD16":["FCGR3A","MS4A7","LILRB1","CX3CR1"],
            "Mac_complement":["C1QA","C1QB","C1QC","APOE","APOC1","MRC1","MARCO","TIMD4"],
            "Mac_DAM":["TREM2","CD9","SPP1","GPNMB","LGALS3"]}
    vn = set(adata.var_names)
    for s,g in sigs.items():
        print(s, "missing:", [x for x in g if x not in vn] or "none")

    print("\n"+"="*60, "\nSTEP 3  patient vs sample\n", "="*60)
    print("n patient:", adata.obs[PATIENT_COL].nunique(), "| n sample:", adata.obs[SAMPLE_COL].nunique())
    print("samples per patient:\n", adata.obs.drop_duplicates(SAMPLE_COL).groupby(PATIENT_COL)[SAMPLE_COL].nunique())

    print("\n"+"="*60, "\nSTEP 4  pairing patient x tissue\n", "="*60)
    ct = pd.crosstab(adata.obs[PATIENT_COL], adata.obs[TISSUE_COL])
    print(ct)
    print("tissues per patient (value counts):\n", ct.gt(0).sum(1).value_counts())

    print("\n"+"="*60, "\nSTEP 5  sample-level n per stage x tissue\n", "="*60)
    samp = adata.obs.drop_duplicates(SAMPLE_COL)
    print("StageGroup x Tissue (n samples):\n", samp.groupby([FIBROSIS_COL, TISSUE_COL]).size().unstack(fill_value=0))
    print("\nStageSep x Tissue (n samples):\n", samp.groupby([STAGESEP_COL, TISSUE_COL]).size().unstack(fill_value=0))
    print("\nstages per patient (should be 1 if patient-level):\n", adata.obs.groupby(PATIENT_COL)[FIBROSIS_COL].nunique().value_counts())
    print("\npatient x StageGroup:\n", pd.crosstab(adata.obs[PATIENT_COL], adata.obs[FIBROSIS_COL]).gt(0).sum(0))

    print("\n"+"="*60, "\nSTEP 6  upstream QC / doublet / hashing traces\n", "="*60)
    print("HTO_classification.global:\n", adata.obs["HTO_classification.global"].value_counts(dropna=False))
    print("\nLIVER HTO global:\n", adata.obs.loc[adata.obs[TISSUE_COL]=="LIVER","HTO_classification.global"].value_counts(dropna=False))
    print("percent.mt  max:", pd.to_numeric(adata.obs["percent.mt"], errors="coerce").max())
    print("percent.mt  median:", pd.to_numeric(adata.obs["percent.mt"], errors="coerce").median())
    print("percent.hb  max:", pd.to_numeric(adata.obs["percent.hb"], errors="coerce").max())
    print("nCount_RNA median:", adata.obs["nCount_RNA"].median(), "| nFeature_RNA median:", adata.obs["nFeature_RNA"].median())

    print("\n"+"="*60, "\nSTEP 7  missing mito/hb\n", "="*60)
    for col in ["percent.mt","percent.hb"]:
        s = pd.to_numeric(adata.obs[col], errors="coerce"); na = s.isna()
        print(f"{col}: {int(na.sum())} missing | by tissue:", adata.obs.loc[na, TISSUE_COL].value_counts().to_dict())

    print("\n"+"="*60, "\nSTEP 8  embeddings / neighbours provenance\n", "="*60)
    for k in adata.obsm: print(f"obsm['{k}'] shape {adata.obsm[k].shape} (n_obs={adata.n_obs})")
    print("uns['neighbors_subcluster']:", adata.uns.get("neighbors_subcluster"))
    print("rank_genes_groups params:", adata.uns.get("rank_genes_groups",{}).get("params"))

text = buf.getvalue()
print(text)
with open("/Users/apple/Downloads/section2_6_outputs/diagnostics_full.txt","w",encoding="utf-8") as f:
    f.write(text)
print("\n已保存到 diagnostics_full.txt")

In [ ]:
import pandas as pd
hb = pd.to_numeric(adata.obs["percent.hb"], errors="coerce")
for t in [1, 5, 10, 20]:
    print(f"percent.hb > {t}%: {(hb > t).sum()} cells",
          "| by tissue:", adata.obs.loc[hb > t, "Tissue"].value_counts().to_dict())
print("total cells:", adata.n_obs)

In [ ]:
import io, contextlib
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

PATIENT="Patient_ID"; TISSUE="Tissue"; DIAB="Diabetic"; CT="cell_type_with_cluster"
SCORES=["CD14_monocyte_score","CD16_monocyte_score",
        "Macrophage_complement_score","Disease_associated_macrophage_score"]

obs = adata.obs.copy()

buf = io.StringIO()
with contextlib.redirect_stdout(buf):

    # ---- 0. 糖尿病在病人层面的分布(确认每个病人只有一个状态) ----
    print("="*60,"\n0. Diabetic status per patient\n","="*60)
    print("stages per patient (should be 1):",
          obs.groupby(PATIENT)[DIAB].nunique().value_counts().to_dict())
    pat_diab = obs.drop_duplicates(PATIENT).set_index(PATIENT)[DIAB]
    print("patients by Diabetic:", pat_diab.value_counts().to_dict())
    print("\nDiabetic x Tissue (n patients):")
    print(obs.drop_duplicates([PATIENT,TISSUE]).groupby([DIAB,TISSUE]).size().unstack(fill_value=0))

    # ---- 1. 细胞类型占比:每个 病人×组织 内三大群的比例 ----
    print("\n"+"="*60,"\n1. Cell-type proportion by Diabetic (patient-level)\n","="*60)
    comp = (obs.groupby([PATIENT,TISSUE,CT]).size()
              .groupby(level=[0,1]).apply(lambda s: s/s.sum())
              .rename("prop").reset_index())
    comp = comp.merge(pat_diab.rename("Diab"), left_on=PATIENT, right_index=True)
    for tis in comp[TISSUE].unique():
        for ct in comp[CT].unique():
            sub = comp[(comp[TISSUE]==tis)&(comp[CT]==ct)]
            a = sub[sub.Diab=="Yes"]["prop"]; b = sub[sub.Diab=="No"]["prop"]
            if len(a)>=2 and len(b)>=2:
                u,p = mannwhitneyu(a,b)
                print(f"{tis:6s} {ct:22s} Yes med={a.median():.3f}(n={len(a)}) "
                      f"No med={b.median():.3f}(n={len(b)})  p={p:.3f}")

    # ---- 2. signature score:每个 病人×组织 的平均分,两组比较 ----
    print("\n"+"="*60,"\n2. Signature scores by Diabetic (patient-level)\n","="*60)
    sc_df = (obs.groupby([PATIENT,TISSUE])[SCORES].mean().reset_index()
               .merge(pat_diab.rename("Diab"), left_on=PATIENT, right_index=True))
    for tis in sc_df[TISSUE].unique():
        for s in SCORES:
            sub = sc_df[sc_df[TISSUE]==tis]
            a = sub[sub.Diab=="Yes"][s]; b = sub[sub.Diab=="No"][s]
            if len(a)>=2 and len(b)>=2:
                u,p = mannwhitneyu(a,b)
                print(f"{tis:6s} {s:36s} Yes={a.median():.3f}(n={len(a)}) "
                      f"No={b.median():.3f}(n={len(b)})  p={p:.3f}")

text = buf.getvalue(); print(text)
open("/Users/apple/Downloads/section2_6_outputs/diabetic_analysis.txt","w",encoding="utf-8").write(text)
print("\n已保存 diabetic_analysis.txt")

In [ ]:
import io, contextlib
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

PATIENT="Patient_ID"; TISSUE="Tissue"; DIAB="Diabetic"; CT="cell_type_with_cluster"
SCORES=["CD14_monocyte_score","CD16_monocyte_score",
        "Macrophage_complement_score","Disease_associated_macrophage_score"]

obs = adata.obs.copy()
pat_diab = obs.drop_duplicates(PATIENT).set_index(PATIENT)[DIAB]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):

    print("="*60,"\n0. Diabetic status per patient\n","="*60)
    print("stages per patient (should be 1):",
          obs.groupby(PATIENT, observed=True)[DIAB].nunique().value_counts().to_dict())
    print("patients by Diabetic:", pat_diab.value_counts().to_dict())
    print("\nDiabetic x Tissue (n patients):")
    print(obs.drop_duplicates([PATIENT,TISSUE])
             .groupby([DIAB,TISSUE], observed=True).size().unstack(fill_value=0))

    # ---- 1. 细胞类型占比(用 transform 算分母,避免 reset_index 冲突) ----
    print("\n"+"="*60,"\n1. Cell-type proportion by Diabetic (patient-level)\n","="*60)
    cnt = (obs.groupby([PATIENT,TISSUE,CT], observed=True)
              .size().rename("n").reset_index())
    cnt["prop"] = cnt["n"] / cnt.groupby([PATIENT,TISSUE], observed=True)["n"].transform("sum")
    comp = cnt.merge(pat_diab.rename("Diab"), left_on=PATIENT, right_index=True)
    for tis in comp[TISSUE].unique():
        for ct in comp[CT].unique():
            sub = comp[(comp[TISSUE]==tis)&(comp[CT]==ct)]
            a = sub[sub.Diab=="Yes"]["prop"]; b = sub[sub.Diab=="No"]["prop"]
            if len(a)>=2 and len(b)>=2:
                u,p = mannwhitneyu(a,b)
                print(f"{tis:6s} {ct:22s} Yes med={a.median():.3f}(n={len(a)}) "
                      f"No med={b.median():.3f}(n={len(b)})  p={p:.3f}")

    # ---- 2. signature score:每个 病人×组织 的平均分,两组比较 ----
    print("\n"+"="*60,"\n2. Signature scores by Diabetic (patient-level)\n","="*60)
    sc_df = (obs.groupby([PATIENT,TISSUE], observed=True)[SCORES].mean().reset_index()
               .merge(pat_diab.rename("Diab"), left_on=PATIENT, right_index=True))
    for tis in sc_df[TISSUE].unique():
        for s in SCORES:
            sub = sc_df[sc_df[TISSUE]==tis]
            a = sub[sub.Diab=="Yes"][s]; b = sub[sub.Diab=="No"][s]
            if len(a)>=2 and len(b)>=2:
                u,p = mannwhitneyu(a,b)
                print(f"{tis:6s} {s:36s} Yes={a.median():.3f}(n={len(a)}) "
                      f"No={b.median():.3f}(n={len(b)})  p={p:.3f}")

text = buf.getvalue(); print(text)
open("/Users/apple/Downloads/section2_6_outputs/diabetic_analysis.txt","w",encoding="utf-8").write(text)
print("\n已保存 diabetic_analysis.txt")

In [ ]:
import scanpy as sc, pandas as pd
sc.tl.rank_genes_groups(adata, groupby="cell_type_with_cluster",
                        method="wilcoxon", use_raw=False, n_genes=50)
df = sc.get.rank_genes_groups_df(adata, group=None)
df.to_csv("/Users/apple/Downloads/section2_6_outputs/ranked_marker_genes_by_CELLTYPE.csv", index=False)
print(df.head(20))
print("saved.")


In [ ]:
adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata

In [ ]:
import scanpy as sc, pandas as pd
sc.tl.rank_genes_groups(adata, groupby="cell_type_with_cluster",
                        method="wilcoxon", use_raw=False, n_genes=50)
df = sc.get.rank_genes_groups_df(adata, group=None)
df.to_csv("/Users/apple/Downloads/section2_6_outputs/ranked_marker_genes_by_CELLTYPE.csv", index=False)
print(df.head(20))
print("saved.")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

# 1. 读取 h5ad
h5ad_path = "/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad"
adata = sc.read_h5ad(h5ad_path)
obs = adata.obs.copy()

# 2. 建立最小输出文件夹
outdir = Path("/Users/apple/Downloads/minimal_dissertation_tables")
outdir.mkdir(parents=True, exist_ok=True)

print(adata)
print("Output folder:", outdir)

# 3. 保存 obs columns，防止列名不匹配时方便检查
pd.Series(obs.columns, name="obs_columns").to_csv(outdir / "00_obs_columns.csv", index=False)
print("obs columns saved.")

In [ ]:
def pick_col(candidates, required=True, label=""):
    for c in candidates:
        if c in obs.columns:
            print(f"{label}: using '{c}'")
            return c
    if required:
        raise ValueError(f"Cannot find column for {label}. Please check 00_obs_columns.csv.")
    else:
        print(f"{label}: not found")
        return None

tissue_col = pick_col(
    ["Tissue", "tissue", "tissue_source", "source", "Source"],
    label="Tissue"
)

stage_col = pick_col(
    ["StageGroup", "stage_group", "Stage", "stage", "fibrosis_stage_group"],
    label="Stage group"
)

celltype_col = pick_col(
    ["cell_type_with_cluster", "cell_type", "CellType", "annotation", "broad_cell_type"],
    label="Cell type"
)

patient_col = pick_col(
    ["Patient_ID", "patient_id", "patient", "donor_id", "Donor", "donor"],
    required=False,
    label="Patient ID"
)

sample_col = pick_col(
    ["Sample", "sample", "sample_id", "orig.ident", "batch", "Batch"],
    required=False,
    label="Sample ID"
)

cluster_col = pick_col(
    ["leiden_0.5", "leiden_0_5", "leiden", "final_subcluster", "subcluster", "cluster"],
    label="Subcluster"
)

analysis_id_col = patient_col if patient_col is not None else sample_col
print("Analysis ID column:", analysis_id_col)

In [ ]:
# 统一格式
obs[tissue_col] = obs[tissue_col].astype(str)
obs[stage_col] = obs[stage_col].astype(str)
obs[celltype_col] = obs[celltype_col].astype(str)

if analysis_id_col is not None:
    obs[analysis_id_col] = obs[analysis_id_col].astype(str)

# 清理 broad cell type 名称
def clean_celltype(x):
    x = str(x)
    x_low = x.lower()
    if "cd14" in x_low:
        return "CD14 monocytes"
    elif "cd16" in x_low or "fcgr3a" in x_low:
        return "CD16 monocytes"
    elif "macro" in x_low:
        return "Macrophages"
    else:
        return x

obs["celltype_clean"] = obs[celltype_col].apply(clean_celltype)

# 生成 count table
def count_percent(series, category, denominator):
    counts = series.value_counts(dropna=False)
    df = counts.rename_axis("Group").reset_index(name="n")
    df.insert(0, "Category", category)
    df["% of total cells"] = (df["n"] / denominator * 100).round(1)
    return df

rows = []

# 总体信息
rows.append(pd.DataFrame([
    {
        "Category": "Dataset",
        "Group": "Total cells",
        "n": adata.n_obs,
        "% of total cells": 100.0
    },
    {
        "Category": "Dataset",
        "Group": "Total genes",
        "n": adata.n_vars,
        "% of total cells": np.nan
    }
]))

if analysis_id_col is not None:
    rows.append(pd.DataFrame([
        {
            "Category": "Dataset",
            "Group": f"Unique {analysis_id_col}",
            "n": obs[analysis_id_col].nunique(),
            "% of total cells": np.nan
        }
    ]))

# Tissue / stage / cell type
rows.append(count_percent(obs[tissue_col], "Tissue", adata.n_obs))
rows.append(count_percent(obs[stage_col], "Stage group", adata.n_obs))
rows.append(count_percent(obs["celltype_clean"], "Broad cell type", adata.n_obs))

table1 = pd.concat(rows, ignore_index=True)

# 可选排序
category_order = ["Dataset", "Tissue", "Stage group", "Broad cell type"]
table1["Category"] = pd.Categorical(table1["Category"], categories=category_order, ordered=True)
table1 = table1.sort_values(["Category", "Group"]).reset_index(drop=True)

table1.to_csv(outdir / "Table1_dataset_composition.csv", index=False)

table1

In [ ]:
# C. 生成 Table 3：完整 subcluster composition

# 清理 cluster label，例如 0 -> C0
def clean_cluster_label(x):
    s = str(x)
    if s.startswith("C"):
        return s
    try:
        return "C" + str(int(float(s)))
    except:
        return s

def cluster_sort_key(x):
    s = str(x).replace("C", "")
    try:
        return int(float(s))
    except:
        return 9999

# 确保 cluster 是普通字符串，不是 categorical
obs["cluster_label"] = obs[cluster_col].astype(str).apply(clean_cluster_label)

# cluster counts
cluster_counts = (
    obs["cluster_label"]
    .value_counts()
    .rename_axis("Cluster")
    .reset_index(name="n")
)

cluster_counts = cluster_counts.sort_values(
    "Cluster",
    key=lambda x: x.map(cluster_sort_key)
)

# tissue percentage, row-wise within each cluster
tissue_pct = pd.crosstab(
    obs["cluster_label"],
    obs[tissue_col].astype(str),
    normalize="index"
) * 100

tissue_pct = tissue_pct.round(1)
tissue_pct = tissue_pct.reset_index().rename(columns={"cluster_label": "Cluster"})

# stage percentage, row-wise within each cluster
stage_pct = pd.crosstab(
    obs["cluster_label"],
    obs[stage_col].astype(str),
    normalize="index"
) * 100

stage_pct = stage_pct.round(1)
stage_pct = stage_pct.reset_index().rename(columns={"cluster_label": "Cluster"})

# 合并
table3 = cluster_counts.merge(tissue_pct, on="Cluster", how="left")
table3 = table3.merge(stage_pct, on="Cluster", how="left")

# 加上你论文目前使用的 biological interpretation
identity_map = {
    "C0": "Classical CD14 monocytes",
    "C1": "Lymphoid-contaminated (excluded)",
    "C2": "Lipid-/disease-associated macrophages",
    "C3": "Inflammatory monocytes",
    "C4": "Complement-high resident macrophages",
    "C5": "Non-classical CD16 monocytes",
    "C6": "MRC1+ macrophages",
    "C7": "CD16 monocytes",
    "C8": "Transitional monocytes",
    "C9": "Erythroid-contaminated (excluded)",
    "C10": "Kupffer-like resident macrophages",
    "C11": "Platelet-contaminated (excluded)",
    "C12": "Lipid-associated macrophages (FABP4+)"
}

# 关键修正：转成 object/string，避免 categorical fillna 报错
table3["Identity"] = table3["Cluster"].astype(str).map(identity_map)
table3["Identity"] = table3["Identity"].astype("object").fillna("")

# 排序
table3 = table3.sort_values(
    "Cluster",
    key=lambda x: x.map(cluster_sort_key)
).reset_index(drop=True)

# 保存
table3.to_csv(outdir / "Table3_subcluster_composition.csv", index=False)

# 也保存 counts-only，方便核对，但不一定放论文
pd.crosstab(obs["cluster_label"], obs[tissue_col].astype(str)).to_csv(
    outdir / "subcluster_by_tissue_counts.csv"
)

pd.crosstab(obs["cluster_label"], obs[stage_col].astype(str)).to_csv(
    outdir / "subcluster_by_stagegroup_counts.csv"
)

table3

In [ ]:
checks = []

checks.append(f"adata shape: {adata.n_obs} cells x {adata.n_vars} genes")
checks.append(f"Tissue column: {tissue_col}")
checks.append(f"Stage column: {stage_col}")
checks.append(f"Cell type column: {celltype_col}")
checks.append(f"Cluster column: {cluster_col}")
checks.append(f"Analysis ID column: {analysis_id_col}")

checks.append("\nTissue counts:")
checks.append(str(obs[tissue_col].value_counts()))

checks.append("\nStage group counts:")
checks.append(str(obs[stage_col].value_counts()))

checks.append("\nBroad cell type counts:")
checks.append(str(obs["celltype_clean"].value_counts()))

checks.append("\nSubcluster counts:")
checks.append(str(obs["cluster_label"].value_counts().sort_index(key=lambda x: x.map(cluster_sort_key))))

checks.append("\nTable 1 total cells:")
checks.append(str(table1[table1["Group"] == "Total cells"]))

checks.append("\nTable 3 total subcluster n:")
checks.append(str(table3["n"].sum()))

(outdir / "00_sanity_check.txt").write_text("\n".join(checks))

print("Saved sanity check to:", outdir / "00_sanity_check.txt")

In [ ]:
import shutil

zip_path = "/Users/apple/Downloads/minimal_dissertation_tables.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", outdir)

print("Created:", zip_path)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata

In [ ]:
print("Unique Patient_ID:", adata.obs['Patient_ID'].nunique())
print("Unique Sample:", adata.obs['Sample'].nunique())
print("\nSample count per patient:")
print(adata.obs.groupby('Patient_ID')['Sample'].nunique())
print("\nSample x Tissue crosstab (first rows):")
print(pd.crosstab(adata.obs['Sample'], adata.obs['Tissue']).head(10))

In [ ]:
# ============================================================
# Diagnostic: clarify Patient_ID vs Sample structure (for Methods 2.2)
#
# HOW TO USE:
#   1. Run this in your Jupyter notebook where `adata` is loaded.
#   2. It saves everything to:  patient_sample_structure_report.txt
#   3. Send me that .txt file.
# ============================================================

import pandas as pd
import os

# ---- WHERE TO SAVE (edit this path if you want a different folder) ----
OUTPUT_DIR = "/Users/apple/Downloads/section2_6_outputs"   # <-- change if needed
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "patient_sample_structure_report.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)

obs = adata.obs
lines = []


def w(text=""):
    """write a line to the report (and show it in the notebook)"""
    print(text)
    lines.append(str(text))


w("=" * 60)
w("1. HOW MANY PATIENTS AND SAMPLES?")
w("=" * 60)
w(f"Unique Patient_ID : {obs['Patient_ID'].nunique()}")
w(f"Unique Sample     : {obs['Sample'].nunique()}")
w(f"Total cells       : {adata.n_obs}")

w()
w("=" * 60)
w("2. LIST OF ALL Patient_ID VALUES")
w("=" * 60)
w(sorted(obs['Patient_ID'].astype(str).unique()))

w()
w("=" * 60)
w("3. LIST OF ALL Sample VALUES")
w("=" * 60)
w(sorted(obs['Sample'].astype(str).unique()))

w()
w("=" * 60)
w("4. HOW MANY Sample PER Patient_ID?")
w("=" * 60)
per_patient = obs.groupby('Patient_ID', observed=True)['Sample'].nunique().sort_index()
w(per_patient.to_string())
w()
w("Distribution (how many patients have N samples):")
w(per_patient.value_counts().sort_index().to_string())

w()
w("=" * 60)
w("5. WHICH Sample(s) BELONG TO EACH Patient_ID?")
w("=" * 60)
mapping = obs.groupby('Patient_ID', observed=True)['Sample'].apply(
    lambda x: sorted(set(x.astype(str)))
).sort_index()
for pid, samples in mapping.items():
    w(f"{str(pid):12} -> {samples}")

w()
w("=" * 60)
w("6. Sample x Tissue CROSSTAB")
w("   (does one Sample span several tissues, or map to one tissue?)")
w("=" * 60)
ct = pd.crosstab(obs['Sample'], obs['Tissue'])
w(ct.to_string())

w()
w("=" * 60)
w("7. Patient_ID x Tissue CROSSTAB (confirm fully paired design)")
w("=" * 60)
ct2 = pd.crosstab(obs['Patient_ID'], obs['Tissue'])
w(ct2.to_string())
w()
w(f"Patients with cells in ALL 4 compartments: "
  f"{int((ct2 > 0).all(axis=1).sum())} out of {ct2.shape[0]}")

w()
w("=" * 60)
w("8. IS 10113-1 / 10113-2 (and 10291-2) ONE PATIENT OR TWO?")
w("=" * 60)
suspects = [p for p in obs['Patient_ID'].astype(str).unique() if '-' in str(p)]
w(f"Patient_IDs containing a '-' suffix: {sorted(suspects)}")
if suspects:
    sub = obs[obs['Patient_ID'].astype(str).isin(suspects)]
    w()
    w("Cells per (Patient_ID, StageGroup):")
    w(sub.groupby(['Patient_ID', 'StageGroup'], observed=True)
         .size().rename('n_cells').to_string())
    w()
    w("Their Sample IDs:")
    w(sub.groupby('Patient_ID', observed=True)['Sample']
         .apply(lambda x: sorted(set(x.astype(str)))).to_string())

w()
w("=" * 60)
w("9. Patient_ID x StageGroup (confirm 13 / 6 / 1 split)")
w("=" * 60)
ct3 = pd.crosstab(obs['Patient_ID'], obs['StageGroup'])
w(ct3.to_string())
w()
w("Patients per stage group:")
w((ct3 > 0).sum(axis=0).to_string())

w()
w("=" * 60)
w("DONE")
w("=" * 60)

# ---- SAVE TO FILE ----
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print()
print(f"Report saved to: {OUTPUT_FILE}")
print("Send me this .txt file.")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata


In [ ]:
# ==============================================================================
# SUPPLEMENTARY ANALYSIS
# Within-tissue stage comparisons of signature scores
#
# WHY: The main stage analysis (Results 3.5) pooled all four compartments, so
#      the adipose-dominated signal may mask what happens inside the liver.
#      This script asks: WITHIN each tissue, do the signatures differ between
#      early (F0_1) and progressive (F2_3) fibrosis?
#
# HOW TO USE:
#   1. Run in the Jupyter notebook where `adata` is loaded.
#   2. Outputs csv files + figures to OUTPUT_DIR.
#   3. Send me the csv files (and the figures if you can).
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ---- SETTINGS ----------------------------------------------------------------
OUTPUT_DIR = "/Users/apple/Downloads/section2_6_outputs"   # <-- change if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

SIGNATURES = [
    "CD14_monocyte_score",
    "CD16_monocyte_score",
    "Macrophage_complement_score",
    "Disease_associated_macrophage_score",
]
TISSUE_KEY   = "Tissue"
STAGE_KEY    = "StageGroup"
PATIENT_KEY  = "Patient_ID"
CELLTYPE_KEY = "cell_type_with_cluster"
CLUSTER_KEY  = "leiden_0.5"

obs = adata.obs.copy()

# sanity check that the signature columns exist
missing = [s for s in SIGNATURES if s not in obs.columns]
if missing:
    raise KeyError(f"Signature columns not found in adata.obs: {missing}")

print("Tissues :", sorted(obs[TISSUE_KEY].astype(str).unique()))
print("Stages  :", sorted(obs[STAGE_KEY].astype(str).unique()))
print()


# ==============================================================================
# HELPER: patient-level aggregation + Mann-Whitney (F0_1 vs F2_3)
# ==============================================================================
def compare_stage_within(df, label, group_col=None):
    """
    df must contain: PATIENT_KEY, STAGE_KEY, and the signature columns.
    Aggregates to patient level (mean per patient), then compares F0_1 vs F2_3
    with a Mann-Whitney U test. H (n=1 patient) is excluded from testing.
    """
    rows = []
    for sig in SIGNATURES:
        # patient-level means
        pat = (df.groupby([PATIENT_KEY, STAGE_KEY], observed=True)[sig]
                 .mean().reset_index())
        a = pat.loc[pat[STAGE_KEY] == "F0_1", sig].dropna().values
        b = pat.loc[pat[STAGE_KEY] == "F2_3", sig].dropna().values
        h = pat.loc[pat[STAGE_KEY] == "H",    sig].dropna().values

        if len(a) >= 2 and len(b) >= 2:
            stat, p = mannwhitneyu(a, b, alternative="two-sided")
        else:
            stat, p = np.nan, np.nan

        rows.append({
            "group": label,
            "signature": sig,
            "n_patients_F0_1": len(a),
            "n_patients_F2_3": len(b),
            "n_patients_H": len(h),
            "mean_F0_1": np.mean(a) if len(a) else np.nan,
            "mean_F2_3": np.mean(b) if len(b) else np.nan,
            "mean_H": np.mean(h) if len(h) else np.nan,
            "median_F0_1": np.median(a) if len(a) else np.nan,
            "median_F2_3": np.median(b) if len(b) else np.nan,
            "direction": ("F0_1 > F2_3" if (len(a) and len(b) and np.mean(a) > np.mean(b))
                          else "F2_3 > F0_1" if (len(a) and len(b)) else "NA"),
            "MannWhitney_U": stat,
            "p_raw": p,
        })
    return pd.DataFrame(rows)


# ==============================================================================
# 1. WITHIN EACH TISSUE: stage comparison (ALL cells)
# ==============================================================================
print("=" * 70)
print("1. WITHIN-TISSUE STAGE COMPARISON  (all monocytes + macrophages)")
print("=" * 70)

res = []
for t in sorted(obs[TISSUE_KEY].astype(str).unique()):
    sub = obs[obs[TISSUE_KEY].astype(str) == t]
    res.append(compare_stage_within(sub, label=t))
res_all = pd.concat(res, ignore_index=True)

# BH correction across all tests in this family
ok = res_all["p_raw"].notna()
res_all.loc[ok, "fdr_bh"] = multipletests(res_all.loc[ok, "p_raw"], method="fdr_bh")[1]

print(res_all[["group", "signature", "mean_F0_1", "mean_F2_3",
               "direction", "p_raw", "fdr_bh"]].to_string(index=False))
res_all.to_csv(f"{OUTPUT_DIR}/within_tissue_stage_ALLCELLS.csv", index=False)
print(f"\n-> saved within_tissue_stage_ALLCELLS.csv\n")


# ==============================================================================
# 2. WITHIN EACH TISSUE: stage comparison (MACROPHAGES ONLY)
#    DAM describes a macrophage state, so monocytes may dilute the signal.
# ==============================================================================
print("=" * 70)
print("2. WITHIN-TISSUE STAGE COMPARISON  (macrophages only)")
print("=" * 70)

mac_mask = obs[CELLTYPE_KEY].astype(str).str.contains("Macrophage", case=False, na=False)
obs_mac = obs[mac_mask]
print(f"Macrophages: {len(obs_mac)} cells")
print(obs_mac[TISSUE_KEY].value_counts().to_string())
print()

res_m = []
for t in sorted(obs_mac[TISSUE_KEY].astype(str).unique()):
    sub = obs_mac[obs_mac[TISSUE_KEY].astype(str) == t]
    # need at least a few patients per group to be meaningful
    res_m.append(compare_stage_within(sub, label=f"{t} (macrophages)"))
res_mac = pd.concat(res_m, ignore_index=True)

ok = res_mac["p_raw"].notna()
res_mac.loc[ok, "fdr_bh"] = multipletests(res_mac.loc[ok, "p_raw"], method="fdr_bh")[1]

print(res_mac[["group", "signature", "mean_F0_1", "mean_F2_3",
               "direction", "p_raw", "fdr_bh"]].to_string(index=False))
res_mac.to_csv(f"{OUTPUT_DIR}/within_tissue_stage_MACROPHAGES.csv", index=False)
print(f"\n-> saved within_tissue_stage_MACROPHAGES.csv\n")


# ==============================================================================
# 3. LIVER-ONLY DETAIL: subcluster composition by stage
#    (which macrophage subclusters change with stage inside the liver?)
# ==============================================================================
print("=" * 70)
print("3. LIVER ONLY: subcluster composition by fibrosis stage")
print("=" * 70)

liver = obs[obs[TISSUE_KEY].astype(str) == "LIVER"]
print(f"Liver cells: {len(liver)}")

# counts
liver_ct = pd.crosstab(liver[CLUSTER_KEY], liver[STAGE_KEY])
# % within each stage (i.e. what fraction of that stage's liver cells is this cluster)
liver_pct = pd.crosstab(liver[CLUSTER_KEY], liver[STAGE_KEY], normalize="columns") * 100

print("\nCell counts (subcluster x stage):")
print(liver_ct.to_string())
print("\nPercentage of each stage's liver cells (column %):")
print(liver_pct.round(1).to_string())

liver_ct.to_csv(f"{OUTPUT_DIR}/liver_subcluster_by_stage_counts.csv")
liver_pct.round(2).to_csv(f"{OUTPUT_DIR}/liver_subcluster_by_stage_percent.csv")
print(f"\n-> saved liver_subcluster_by_stage_counts.csv / _percent.csv\n")

# patient-level proportion of each subcluster within liver, compared across stage
print("Patient-level liver subcluster proportions, F0_1 vs F2_3:")
liver_prop_rows = []
for cl in sorted(liver[CLUSTER_KEY].astype(str).unique(), key=lambda x: int(x)):
    per_pat = (liver.assign(is_cl=(liver[CLUSTER_KEY].astype(str) == cl))
                    .groupby([PATIENT_KEY, STAGE_KEY], observed=True)["is_cl"]
                    .mean().reset_index())
    a = per_pat.loc[per_pat[STAGE_KEY] == "F0_1", "is_cl"].values
    b = per_pat.loc[per_pat[STAGE_KEY] == "F2_3", "is_cl"].values
    if len(a) >= 2 and len(b) >= 2:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
    else:
        stat, p = np.nan, np.nan
    liver_prop_rows.append({
        "subcluster": f"C{cl}",
        "mean_prop_F0_1_pct": 100 * np.mean(a) if len(a) else np.nan,
        "mean_prop_F2_3_pct": 100 * np.mean(b) if len(b) else np.nan,
        "MannWhitney_U": stat, "p_raw": p,
    })
liver_prop = pd.DataFrame(liver_prop_rows)
ok = liver_prop["p_raw"].notna()
liver_prop.loc[ok, "fdr_bh"] = multipletests(liver_prop.loc[ok, "p_raw"], method="fdr_bh")[1]
print(liver_prop.round(3).to_string(index=False))
liver_prop.to_csv(f"{OUTPUT_DIR}/liver_subcluster_proportion_stage_test.csv", index=False)
print(f"\n-> saved liver_subcluster_proportion_stage_test.csv\n")


# ==============================================================================
# 4. FIGURE: DAM (and all signatures) by stage, faceted by tissue
# ==============================================================================
print("=" * 70)
print("4. FIGURES")
print("=" * 70)

# patient/tissue-level table for plotting
plot_df = (obs.groupby([PATIENT_KEY, TISSUE_KEY, STAGE_KEY], observed=True)[SIGNATURES]
              .mean().reset_index())
plot_df.to_csv(f"{OUTPUT_DIR}/patient_tissue_stage_signature_means.csv", index=False)

tissue_order = [t for t in ["PBMC", "LIVER", "VAT", "SAT"]
                if t in plot_df[TISSUE_KEY].astype(str).unique()]
stage_order = [s for s in ["F0_1", "F2_3", "H"]
               if s in plot_df[STAGE_KEY].astype(str).unique()]

for sig in SIGNATURES:
    fig, axes = plt.subplots(1, len(tissue_order),
                             figsize=(4 * len(tissue_order), 4), sharey=True)
    if len(tissue_order) == 1:
        axes = [axes]
    for ax, t in zip(axes, tissue_order):
        d = plot_df[plot_df[TISSUE_KEY].astype(str) == t]
        sns.boxplot(data=d, x=STAGE_KEY, y=sig, order=stage_order, ax=ax,
                    showfliers=False)
        sns.stripplot(data=d, x=STAGE_KEY, y=sig, order=stage_order, ax=ax,
                      color="black", size=4, alpha=0.6)
        ax.set_title(t)
        ax.set_xlabel("")
        if ax is not axes[0]:
            ax.set_ylabel("")
    fig.suptitle(f"{sig} by fibrosis stage, within each tissue", y=1.02)
    plt.tight_layout()
    fname = f"{OUTPUT_DIR}/boxplot_{sig}_by_stage_within_tissue.png"
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"-> saved {os.path.basename(fname)}")

print()
print("=" * 70)
print("KEY QUESTION TO CHECK IN THE OUTPUT:")
print("  In LIVER, is the Disease_associated_macrophage_score")
print("  higher in F2_3 (as advanced-fibrosis literature predicts)")
print("  or higher in F0_1 (as the pooled analysis showed)?")
print("=" * 70)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("Send me the .csv files.")



In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata

In [ ]:
print(adata.obs.columns.tolist())

In [ ]:
# 检查 uns 里有没有额外的元数据
print(adata.uns.keys())

# 检查是否有其他 obs 层级的信息
print(adata.obsm.keys())

In [ ]:
import pandas as pd

obs = adata.obs
# 每个病人一行(取该病人的临床信息,病人内应当一致)
pat = (obs.groupby('Patient_ID', observed=True)
          .agg(StageGroup=('StageGroup','first'),
               Stage=('Stage','first'),
               StageSep=('StageSep','first'),
               Diabetic=('Diabetic','first'),
               n_cells=('Tissue','size'))
          .reset_index())

print("=== 每个病人的临床信息 ===")
print(pat.to_string(index=False))

print("\n=== 按 StageGroup 汇总 ===")
print("患者数:")
print(pat['StageGroup'].value_counts().to_string())
print("\n糖尿病 x 分期组:")
print(pd.crosstab(pat['StageGroup'], pat['Diabetic']).to_string())
print("\n详细纤维化分期 x 分期组:")
print(pd.crosstab(pat['StageGroup'], pat['Stage']).to_string())

pat.to_csv('patient_clinical_available.csv', index=False)
print("\n已保存 patient_clinical_available.csv")

In [ ]:
h = adata.obs[adata.obs['StageGroup']=='H']
print(h['Tissue'].value_counts())
print(h.groupby(['Tissue','cell_type_with_cluster'], observed=True).size())

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

# Scrublet 需要原始计数
adb = adata.copy()
adb.X = adb.layers['counts_RNA'].copy()

# 按测序批次分别跑(doublet 是在同一个 run 内形成的)
sc.pp.scrublet(adb, batch_key='Sample')     # 旧版 scanpy: sc.external.pp.scrublet

adata.obs['doublet_score']     = adb.obs['doublet_score'].values
adata.obs['predicted_doublet'] = adb.obs['predicted_doublet'].values

# 按亚群汇总
summ = (adata.obs.groupby('leiden_0.5', observed=True)
        .agg(n_cells=('doublet_score','size'),
             median_score=('doublet_score','median'),
             pct_doublet=('predicted_doublet', lambda x: 100*np.mean(x)))
        .round(3))
print(summ.to_string())
summ.to_csv('doublet_scores_by_subcluster.csv')

print("\n整体预测 doublet 比例: %.2f%%" % (100*adata.obs['predicted_doublet'].mean()))

In [ ]:
!pip install scikit-image

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata

In [ ]:
import skimage
print(skimage.__version__)


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

# Scrublet 需要原始计数
adb = adata.copy()
adb.X = adb.layers['counts_RNA'].copy()

# 按测序批次分别跑(doublet 是在同一个 run 内形成的)
sc.pp.scrublet(adb, batch_key='Sample')     # 旧版 scanpy: sc.external.pp.scrublet

adata.obs['doublet_score']     = adb.obs['doublet_score'].values
adata.obs['predicted_doublet'] = adb.obs['predicted_doublet'].values

# 按亚群汇总
summ = (adata.obs.groupby('leiden_0.5', observed=True)
        .agg(n_cells=('doublet_score','size'),
             median_score=('doublet_score','median'),
             pct_doublet=('predicted_doublet', lambda x: 100*np.mean(x)))
        .round(3))
print(summ.to_string())
summ.to_csv('doublet_scores_by_subcluster.csv')

print("\n整体预测 doublet 比例: %.2f%%" % (100*adata.obs['predicted_doublet'].mean()))


In [ ]:
counts = adata.obs['Sample'].value_counts().sort_values()
print(counts.head(15))
print("\n批次总数:", len(counts))
print("细胞数 < 50 的批次:", (counts < 50).sum())

In [ ]:
import scanpy as sc, numpy as np, pandas as pd

adb = adata.copy()
adb.X = adb.layers['counts_RNA'].copy()

scores = pd.Series(np.nan, index=adata.obs_names, dtype=float)
preds  = pd.Series(np.nan, index=adata.obs_names, dtype=float)
skipped = []

for s in adb.obs['Sample'].unique():
    sub = adb[adb.obs['Sample'] == s].copy()
    n = sub.n_obs
    if n < 30:                       # 太小,跳过
        skipped.append((s, n)); continue
    npc = min(30, n // 3, sub.n_vars - 1)   # 自动适配主成分数
    npc = max(npc, 2)
    try:
        sc.pp.scrublet(sub, n_prin_comps=npc, random_state=0, verbose=False)
        scores[sub.obs_names] = sub.obs['doublet_score'].values
        preds[sub.obs_names]  = sub.obs['predicted_doublet'].astype(float).values
    except Exception as e:
        skipped.append((s, n)); print(f"skip {s} (n={n}): {type(e).__name__}")

adata.obs['doublet_score'] = scores.values
adata.obs['predicted_doublet'] = preds.values

print("\n跳过的批次:", skipped)
print("成功计算的细胞: %d / %d" % (scores.notna().sum(), adata.n_obs))

# 按亚群汇总
summ = (adata.obs.dropna(subset=['doublet_score'])
        .groupby('leiden_0.5', observed=True)
        .agg(n_cells=('doublet_score','size'),
             median_score=('doublet_score','median'),
             mean_score=('doublet_score','mean'),
             pct_predicted_doublet=('predicted_doublet', lambda x: 100*np.mean(x)))
        .round(3))
print("\n=== 各亚群 doublet 指标 ===")
print(summ.to_string())
summ.to_csv('doublet_by_subcluster.csv')

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata

In [ ]:
sigs = {
 'CD14_monocyte': ['CD14','FCN1','S100A8','S100A9','VCAN','CCR2'],
 'CD16_monocyte': ['FCGR3A','MS4A7','LILRB1','CX3CR1'],
 'Macrophage_complement': ['C1QA','C1QB','C1QC','APOE','APOC1'],
 'Disease_associated_macrophage': ['TREM2','CD9','SPP1','GPNMB','LGALS3'],
}
for name, genes in sigs.items():
    missing = [g for g in genes if g not in adata.var_names]
    print(f"{name}: 缺失 {missing if missing else '無'}")

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

sig_cols = ['CD14_monocyte_score','CD16_monocyte_score',
            'Macrophage_complement_score','Disease_associated_macrophage_score']
tissues = ['PBMC','LIVER','VAT','SAT']

# 病人/组织层面均值
pt = (adata.obs.groupby(['Patient_ID','Tissue'], observed=True)[sig_cols]
      .mean().reset_index())

# --- Friedman:四个组织的整体差异 ---
rows=[]
for s in sig_cols:
    wide = pt.pivot(index='Patient_ID', columns='Tissue', values=s)[tissues].dropna()
    chi2, p = friedmanchisquare(*[wide[t].values for t in tissues])
    rows.append({'signature': s, 'n_patients': wide.shape[0],
                 'chi2': round(chi2,2), 'p_raw': p})
res = pd.DataFrame(rows)
res['FDR'] = multipletests(res['p_raw'], method='fdr_bh')[1]
print("=== Friedman(4 signatures × 4 tissues, paired)===")
print(res.to_string(index=False))

# --- 事后两两比较:Wilcoxon signed-rank ---
post=[]
for s in sig_cols:
    wide = pt.pivot(index='Patient_ID', columns='Tissue', values=s)[tissues].dropna()
    for i in range(len(tissues)):
        for j in range(i+1, len(tissues)):
            a,b = tissues[i], tissues[j]
            stat,p = wilcoxon(wide[a], wide[b])
            post.append({'signature': s, 'comparison': f'{a} vs {b}',
                         'median_diff': round(np.median(wide[a]-wide[b]),3), 'p_raw': p})
post = pd.DataFrame(post)
post['FDR'] = multipletests(post['p_raw'], method='fdr_bh')[1]   # 24 个比较一起校正
print("\n=== 事后 Wilcoxon signed-rank(BH across 24 comparisons)===")
print(post.round(4).to_string(index=False))

res.to_csv('friedman_tissue.csv', index=False)
post.to_csv('posthoc_wilcoxon_tissue.csv', index=False)

In [ ]:
import pandas as pd
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

sig_cols = ['CD14_monocyte_score','CD16_monocyte_score',
            'Macrophage_complement_score','Disease_associated_macrophage_score']

# 先按病人平均:每个病人 1 个值(跨四个组织的所有细胞)
pat = (adata.obs.groupby('Patient_ID', observed=True)
       .agg(StageGroup=('StageGroup','first'),
            **{c:(c,'mean') for c in sig_cols})
       .reset_index())

print("各组病人数:")
print(pat['StageGroup'].value_counts().to_string())

rows=[]
for s in sig_cols:
    groups=[g[s].values for _,g in pat.groupby('StageGroup', observed=True)]
    h,p = kruskal(*groups)
    rows.append({'signature': s, 'H': round(h,2), 'p_raw': p})
res = pd.DataFrame(rows)
res['FDR'] = multipletests(res['p_raw'], method='fdr_bh')[1]

print("\n=== 分期比较(病人层面聚合,BH across 4 signatures)===")
print(res.round(4).to_string(index=False))
res.to_csv('stage_kruskal_patientlevel.csv', index=False)

In [ ]:
print(pat.groupby('StageGroup', observed=True)[sig_cols].mean().round(3).to_string())

In [ ]:
# 原来:patient_signature_scores 是 patient×tissue
# 改成按病人平均
pat_plot = adata.obs.groupby('Patient_ID', observed=True).agg(
    StageGroup=('StageGroup','first'),
    **{c:(c,'mean') for c in sig_cols}).reset_index()

for score in sig_cols:
    plt.figure(figsize=(6,4))
    pat_plot.boxplot(column=score, by='StageGroup', rot=45)
    ...
    

In [ ]:
import numpy as np

for score in sig_cols:
    fig, ax = plt.subplots(figsize=(6,4))
    groups = ['F0_1','F2_3','H']
    data = [pat_plot.loc[pat_plot['StageGroup']==g, score].values for g in groups]
    ax.boxplot(data, labels=groups)
    # 叠加散点(带轻微横向抖动)
    for i, d in enumerate(data, start=1):
        x = np.random.normal(i, 0.05, size=len(d))
        ax.scatter(x, d, alpha=0.7, s=25, color='steelblue', zorder=3)
    ax.set_ylabel(score.replace('_',' '))
    ax.set_xlabel("Fibrosis stage group")
    ax.set_title(score.replace('_',' ') + " by stage group")
    plt.tight_layout()
    plt.savefig(f"{outdir}/boxplot_{score}_by_stagegroup_patientlevel.png",
                dpi=300, bbox_inches="tight")
    plt.close()
    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

outdir = "/Users/apple/Downloads/figure5_patient_level"
os.makedirs(outdir, exist_ok=True)

sig_cols = ['CD14_monocyte_score','CD16_monocyte_score',
            'Macrophage_complement_score','Disease_associated_macrophage_score']

# 病人层面聚合(每人 1 个值)
pat_plot = (adata.obs.groupby('Patient_ID', observed=True)
            .agg(StageGroup=('StageGroup','first'),
                 **{c:(c,'mean') for c in sig_cols})
            .reset_index())

groups = ['F0_1','F2_3','H']

for score in sig_cols:
    fig, ax = plt.subplots(figsize=(6,4))
    data = [pat_plot.loc[pat_plot['StageGroup']==g, score].values for g in groups]
    ax.boxplot(data, tick_labels=groups)          # 用新参数名,消除警告
    for i, d in enumerate(data, start=1):
        x = np.random.normal(i, 0.05, size=len(d))
        ax.scatter(x, d, alpha=0.7, s=25, color='steelblue', zorder=3)
    ax.set_ylabel(score.replace('_',' '))
    ax.set_xlabel("Fibrosis stage group")
    ax.set_title(score.replace('_',' ') + " by stage group")
    plt.tight_layout()
    plt.savefig(f"{outdir}/boxplot_{score}_by_stagegroup_patientlevel.png",
                dpi=300, bbox_inches="tight")
    plt.close()

print("已保存到:", outdir)
for f in sorted(os.listdir(outdir)):
    print(" ", f)

In [ ]:
from statsmodels.stats.power import TTestIndPower
d = TTestIndPower().solve_power(nobs1=13, ratio=6/13, alpha=0.05, power=0.8)
print(round(d, 2))


In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

def boot_ci(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    d = [rng.choice(a, len(a), True).mean() - rng.choice(b, len(b), True).mean()
         for _ in range(n)]
    return np.percentile(d, [2.5, 97.5])

# 以 3.6 的组织内比较为例
for t in ['PBMC','LIVER','VAT','SAT']:
    sub = pt[pt['Tissue']==t]          # pt = patient/tissue 层面的表
    for s in sig_cols:
        a = sub.loc[sub['StageGroup']=='F0_1', s].values
        b = sub.loc[sub['StageGroup']=='F2_3', s].values
        if len(a)<2 or len(b)<2: continue
        u,p = mannwhitneyu(a,b)
        rb = 1 - 2*u/(len(a)*len(b))          # rank-biserial 效应量
        lo,hi = boot_ci(a,b)
        print(f"{t:6} {s:36} diff={a.mean()-b.mean():+.3f} "
              f"95%CI [{lo:+.3f}, {hi:+.3f}] rb={rb:+.2f} p={p:.3f}")
        

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

sig_cols = ['CD14_monocyte_score','CD16_monocyte_score',
            'Macrophage_complement_score','Disease_associated_macrophage_score']

# 重建 patient/tissue 表,这次带上 StageGroup
pt = (adata.obs.groupby(['Patient_ID','Tissue'], observed=True)
      .agg(StageGroup=('StageGroup','first'),
           **{c:(c,'mean') for c in sig_cols})
      .reset_index())

def boot_ci(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    d = [rng.choice(a, len(a), True).mean() - rng.choice(b, len(b), True).mean()
         for _ in range(n)]
    return np.percentile(d, [2.5, 97.5])

rows = []
for t in ['PBMC','LIVER','VAT','SAT']:
    sub = pt[pt['Tissue'] == t]
    for s in sig_cols:
        a = sub.loc[sub['StageGroup']=='F0_1', s].values
        b = sub.loc[sub['StageGroup']=='F2_3', s].values
        if len(a) < 2 or len(b) < 2:
            continue
        u, p = mannwhitneyu(a, b)
        rb = 1 - 2*u/(len(a)*len(b))          # rank-biserial 效应量
        lo, hi = boot_ci(a, b)
        rows.append({'tissue': t, 'signature': s,
                     'mean_F0_1': round(a.mean(), 3),
                     'mean_F2_3': round(b.mean(), 3),
                     'diff': round(a.mean()-b.mean(), 3),
                     'CI_low': round(lo, 3), 'CI_high': round(hi, 3),
                     'rank_biserial': round(rb, 2),
                     'p': round(p, 3)})

import pandas as pd
res = pd.DataFrame(rows)
print(res.to_string(index=False))
res.to_csv('effect_sizes_within_compartment.csv', index=False)

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

def boot_ci(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    d = [rng.choice(a, len(a), True).mean() - rng.choice(b, len(b), True).mean()
         for _ in range(n)]
    return np.percentile(d, [2.5, 97.5])

for s in sig_cols:
    a = pat.loc[pat['StageGroup']=='F0_1', s].values
    b = pat.loc[pat['StageGroup']=='F2_3', s].values
    u, p = mannwhitneyu(a, b)
    rb = 1 - 2*u/(len(a)*len(b))
    lo, hi = boot_ci(a, b)
    print(f"{s:36} diff={a.mean()-b.mean():+.3f} 95%CI [{lo:+.3f}, {hi:+.3f}] rb={rb:+.2f}")

In [ ]:
import scanpy as sc, numpy as np, pandas as pd
from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests

sig_cols = ['CD14_monocyte_score','CD16_monocyte_score',
            'Macrophage_complement_score','Disease_associated_macrophage_score']
tissues = ['PBMC','LIVER','VAT','SAT']

sigs = {
 'CD14_monocyte_score': ['CD14','FCN1','S100A8','S100A9','VCAN','CCR2'],
 'CD16_monocyte_score': ['FCGR3A','MS4A7','LILRB1','CX3CR1'],
 'Macrophage_complement_score': ['C1QA','C1QB','C1QC','APOE','APOC1'],
 'Disease_associated_macrophage_score': ['TREM2','CD9','SPP1','GPNMB','LGALS3'],
}

# --- 1. 先量化深度差异(导师也要这个数字) ---
raw = adata.layers['counts_RNA']
tot = np.asarray(raw.sum(axis=1)).ravel()
depth = pd.DataFrame({'Tissue': adata.obs['Tissue'].values, 'counts': tot})
print("=== 各组织每细胞总 counts ===")
print(depth.groupby('Tissue')['counts'].describe()[['25%','50%','75%']].round(0).to_string())

target = int(depth.groupby('Tissue')['counts'].median().min())
print(f"\n降采样目标(最低组织中位数): {target}")

# --- 2. 降采样到共同深度,重新归一化打分 ---
ad = adata.copy()
ad.X = ad.layers['counts_RNA'].copy()
sc.pp.downsample_counts(ad, counts_per_cell=target, random_state=0)
sc.pp.normalize_total(ad, target_sum=1e4)
sc.pp.log1p(ad)

for name, genes in sigs.items():
    sc.tl.score_genes(ad, gene_list=[g for g in genes if g in ad.var_names],
                      score_name=name+'_ds', use_raw=False)

# --- 3. 用降采样后的分数重跑 Friedman ---
ds_cols = [c+'_ds' for c in sig_cols]
pt = (ad.obs.groupby(['Patient_ID','Tissue'], observed=True)[ds_cols]
      .mean().reset_index())

rows=[]
for c, s in zip(sig_cols, ds_cols):
    wide = pt.pivot(index='Patient_ID', columns='Tissue', values=s)[tissues].dropna()
    chi2, p = friedmanchisquare(*[wide[t].values for t in tissues])
    means = wide.mean().round(2).to_dict()
    rows.append({'signature': c, 'chi2': round(chi2,2), 'p_raw': p, **means})
res = pd.DataFrame(rows)
res['FDR'] = multipletests(res['p_raw'], method='fdr_bh')[1]

print("\n=== 降采样后的 Friedman 结果 ===")
print(res.to_string(index=False))
res.to_csv('downsampled_friedman.csv', index=False)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/Users/apple/Downloads/section2_6_outputs/mono_macro_with_subclusters_2_6.h5ad")
adata



In [ ]:
# 1. C10 的真实细胞数(全部)
c10_total = (adata.obs['leiden_0.5'] == '10').sum()
print("C10 全部细胞数:", c10_total)

# 2. C10 中被 Scrublet 打分的细胞数
c10_scored = adata.obs.loc[adata.obs['leiden_0.5']=='10', 'doublet_score'].notna().sum()
print("C10 被 Scrublet 打分的细胞数:", c10_scored)

# 3. 顺便核对 C5 和 C8(也有 ±1 出入)
for c in ['5','8','10']:
    total = (adata.obs['leiden_0.5']==c).sum()
    scored = adata.obs.loc[adata.obs['leiden_0.5']==c,'doublet_score'].notna().sum()
    print(f"C{c}: 全部 {total}, 打分 {scored}, 差 {total-scored}")
    

In [ ]:
skipped_runs = ['GC-WL-9991-LIVER','GC-WL-10203-LIVER','GC-WL-11471-LIVER']
mask = adata.obs['Sample'].isin(skipped_runs) & (adata.obs['leiden_0.5']=='10')
print("被跳过的 run 里的 C10 细胞数:", mask.sum())